In [3]:
# ==========================================
# TASK 2: SEARCH QUERY SPELLING CORRECTOR
# ==========================================

import re
import os


# ==========================================
# Step 1: Load the spelling-error corpus
# ==========================================

files = [
    "spellcheck1.txt",
    "spellcheck2.txt",
    "spellcheck3.txt"
]

all_words = []

for file in files:

    if not os.path.exists(file):
        print("File not found:", file)
        continue

    with open(file, "r", encoding="utf-8", errors="ignore") as f:

        text = f.read()

        # Extract alphabetic words from the file
        words = re.findall(r"[a-zA-Z]+", text.lower())

        all_words.extend(words)


print("Spelling Corpus Loaded")
print("Total words found:", len(all_words))


# ==========================================
# Step 2: Build vocabulary
# ==========================================

vocabulary = set(all_words)

print("Vocabulary Size:", len(vocabulary))


# ==========================================
# Step 3: Edit Distance Function
# ==========================================

def edit_distance(word1, word2):

    # Create matrix
    rows = len(word1) + 1
    cols = len(word2) + 1

    dp = [[0] * cols for _ in range(rows)]

    # First column
    for i in range(rows):
        dp[i][0] = i

    # First row
    for j in range(cols):
        dp[0][j] = j

    # Calculate distance
    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    return dp[-1][-1]


# ==========================================
# Step 4: Find closest matching word
# ==========================================

def correct_word(word):

    word = word.lower()

    # If already present
    if word in vocabulary:
        return word, 0

    # Safety check
    if not vocabulary:
        return word, -1

    # Calculate edit distance
    distances = []

    for candidate in vocabulary:

        distance = edit_distance(word, candidate)

        distances.append((distance, candidate))

    # Sort by distance
    distances.sort()

    closest_distance = distances[0][0]
    closest_word = distances[0][1]

    return closest_word, closest_distance


# ==========================================
# Step 5: Correct Search Query
# ==========================================

def correct_query(query):

    # 4. Tokenize query
    words = re.findall(r"[a-zA-Z]+", query.lower())

    incorrect_words = []
    corrections = []
    corrected_words = []

    for word in words:

        # 5. Identify words not in vocabulary
        if word not in vocabulary:

            incorrect_words.append(word)

            # 6 & 7. Calculate edit distance
            corrected, distance = correct_word(word)

            corrections.append(
                (word, corrected, distance)
            )

            corrected_words.append(corrected)

        else:

            corrected_words.append(word)

    # 8. Create corrected query
    corrected_query = " ".join(corrected_words)

    return incorrect_words, corrections, corrected_query


# ==========================================
# Step 6: Test Multiple Spelling Errors
# ==========================================

test_queries = [
    "machne lerning cours",
    "pythn progrmming",
    "artifical inteligence",
    "computr scince",
    "dat bas"
]


print("\n")
print("=" * 70)
print("MULTIPLE QUERY TESTING")
print("=" * 70)


for query in test_queries:

    incorrect_words, corrections, corrected_query = correct_query(query)

    print("\nOriginal Query:")
    print(query)

    print("\nIncorrect Words:")

    if incorrect_words:
        for word in incorrect_words:
            print("-", word)
    else:
        print("No spelling errors")

    print("\nSuggested Corrections:")

    if corrections:

        for incorrect, correct, distance in corrections:

            print(
                incorrect,
                "->",
                correct,
                "(Edit Distance:",
                distance,
                ")"
            )

    else:
        print("No corrections required")

    print("\nFinal Corrected Query:")
    print(corrected_query)

    print("-" * 70)


# ==========================================
# Step 7: User Search Query
# ==========================================

print("\n")
print("=" * 70)
print("SEARCH QUERY SPELLING CORRECTOR")
print("=" * 70)

user_query = input("\nEnter your search query: ")


# Correct user query
incorrect_words, corrections, corrected_query = correct_query(user_query)


# ==========================================
# Step 8: Display Results
# ==========================================

print("\nOriginal Query:")
print(user_query)


print("\nIncorrect Words:")

if incorrect_words:

    for word in incorrect_words:
        print("-", word)

else:

    print("No spelling errors found")


print("\nSuggested Corrections:")

if corrections:

    for incorrect, correct, distance in corrections:

        print(
            incorrect,
            "->",
            correct,
            "(Edit Distance:",
            distance,
            ")"
        )

else:

    print("No corrections required")


print("\nFinal Corrected Query:")
print(corrected_query)

Spelling Corpus Loaded
Total words found: 48804
Vocabulary Size: 41485


MULTIPLE QUERY TESTING

Original Query:
machne lerning cours

Incorrect Words:
- machne

Suggested Corrections:
machne -> machane (Edit Distance: 1 )

Final Corrected Query:
machane lerning cours
----------------------------------------------------------------------

Original Query:
pythn progrmming

Incorrect Words:
- pythn
- progrmming

Suggested Corrections:
pythn -> path (Edit Distance: 2 )
progrmming -> progreing (Edit Distance: 2 )

Final Corrected Query:
path progreing
----------------------------------------------------------------------

Original Query:
artifical inteligence

Incorrect Words:
No spelling errors

Suggested Corrections:
No corrections required

Final Corrected Query:
artifical inteligence
----------------------------------------------------------------------

Original Query:
computr scince

Incorrect Words:
- computr

Suggested Corrections:
computr -> computer (Edit Distance: 1 )

Final Cor


Enter your search query:  hi hello 



Original Query:
hi hello 

Incorrect Words:
No spelling errors found

Suggested Corrections:
No corrections required

Final Corrected Query:
hi hello


In [4]:
# ==========================================================
# TASK 2: SEARCH QUERY SPELLING CORRECTOR
# ==========================================================

import re
import os


# ==========================================================
# Step 1: Load the spelling-error corpus
# ==========================================================

files = [
    "spellcheck1.txt",
    "spellcheck2.txt",
    "spellcheck3.txt"
]

# Dictionary:
# misspelled word -> correct word
correction_map = {}

# Set of correctly spelled words
vocabulary = set()


for file in files:

    if not os.path.exists(file):
        print("File not found:", file)
        continue

    current_correct_word = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:

        for line in f:

            word = line.strip()

            if not word:
                continue

            # ------------------------------------------
            # $ indicates the correctly spelled word
            # ------------------------------------------

            if word.startswith("$"):

                current_correct_word = word[1:].lower()

                # Add correct word to vocabulary
                vocabulary.add(current_correct_word)

            else:

                # This is a misspelled word
                misspelled_word = word.lower()

                # Connect misspelled word with correct word
                if current_correct_word is not None:

                    correction_map[misspelled_word] = current_correct_word


print("Spelling Corpus Loaded")
print("Correct Words in Vocabulary:", len(vocabulary))
print("Misspelled Words:", len(correction_map))


# ==========================================================
# Step 2: Edit Distance Function
# ==========================================================

def edit_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    dp = [[0] * cols for _ in range(rows)]

    # First column
    for i in range(rows):
        dp[i][0] = i

    # First row
    for j in range(cols):
        dp[0][j] = j

    # Calculate edit distance
    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[-1][-1]


# ==========================================================
# Step 3: Find closest matching word
# ==========================================================

def correct_word(word):

    word = word.lower()

    # Word is already correct
    if word in vocabulary:
        return word, 0

    # If exact misspelling exists in dataset
    if word in correction_map:

        correct = correction_map[word]

        distance = edit_distance(word, correct)

        return correct, distance

    # Otherwise find closest correct word
    if not vocabulary:
        return word, -1

    best_word = None
    best_distance = float("inf")

    for candidate in vocabulary:

        # Small optimization
        if abs(len(word) - len(candidate)) > best_distance:
            continue

        distance = edit_distance(word, candidate)

        if distance < best_distance:

            best_distance = distance
            best_word = candidate

    return best_word, best_distance


# ==========================================================
# Step 4: Correct Search Query
# ==========================================================

def correct_query(query):

    # Tokenize query
    words = re.findall(r"[a-zA-Z]+", query.lower())

    incorrect_words = []
    corrections = []
    corrected_words = []

    for word in words:

        # Check whether word is correct
        if word in vocabulary:

            corrected_words.append(word)

        else:

            incorrect_words.append(word)

            # Find correction
            corrected, distance = correct_word(word)

            corrections.append(
                (word, corrected, distance)
            )

            corrected_words.append(corrected)

    corrected_query = " ".join(corrected_words)

    return (
        incorrect_words,
        corrections,
        corrected_query
    )


# ==========================================================
# Step 5: Test Multiple Spelling Errors
# ==========================================================

test_queries = [

    "machne lerning cours",

    "computr scince",

    "artifical inteligence",

    "definately reccomendation",

    "accomodate recieve"
]


print("\n")
print("=" * 70)
print("MULTIPLE QUERY TESTING")
print("=" * 70)


for query in test_queries:

    incorrect_words, corrections, corrected_query = \
        correct_query(query)

    print("\nOriginal Query:")
    print(query)

    print("\nIncorrect Words:")

    if incorrect_words:

        for word in incorrect_words:
            print("-", word)

    else:

        print("No spelling errors")

    print("\nSuggested Corrections:")

    if corrections:

        for incorrect, correct, distance in corrections:

            print(
                incorrect,
                "->",
                correct,
                "(Edit Distance:",
                distance,
                ")"
            )

    else:

        print("No corrections required")

    print("\nFinal Corrected Query:")
    print(corrected_query)

    print("-" * 70)


# ==========================================================
# Step 6: User Search Query
# ==========================================================

print("\n")
print("=" * 70)
print("SEARCH QUERY SPELLING CORRECTOR")
print("=" * 70)

user_query = input("\nEnter your search query: ")


# Correct user query

incorrect_words, corrections, corrected_query = \
    correct_query(user_query)


# ==========================================================
# Step 7: Display Results
# ==========================================================

print("\nOriginal Query:")
print(user_query)


print("\nIncorrect Words:")

if incorrect_words:

    for word in incorrect_words:
        print("-", word)

else:

    print("No spelling errors found")


print("\nSuggested Corrections:")

if corrections:

    for incorrect, correct, distance in corrections:

        print(
            incorrect,
            "->",
            correct,
            "(Edit Distance:",
            distance,
            ")"
        )

else:

    print("No corrections required")


print("\nFinal Corrected Query:")
print(corrected_query)

Spelling Corpus Loaded
Correct Words in Vocabulary: 7392
Misspelled Words: 36025


MULTIPLE QUERY TESTING

Original Query:
machne lerning cours

Incorrect Words:
- machne
- lerning
- cours

Suggested Corrections:
machne -> machine (Edit Distance: 1 )
lerning -> learning (Edit Distance: 1 )
cours -> courses (Edit Distance: 2 )

Final Corrected Query:
machine learning courses
----------------------------------------------------------------------

Original Query:
computr scince

Incorrect Words:
- computr
- scince

Suggested Corrections:
computr -> computer (Edit Distance: 1 )
scince -> since (Edit Distance: 1 )

Final Corrected Query:
computer since
----------------------------------------------------------------------

Original Query:
artifical inteligence

Incorrect Words:
- artifical
- inteligence

Suggested Corrections:
artifical -> artificial (Edit Distance: 1 )
inteligence -> intelligence (Edit Distance: 1 )

Final Corrected Query:
artificial intelligence
--------------------------


Enter your search query:  hllo



Original Query:
hllo

Incorrect Words:
- hllo

Suggested Corrections:
hllo -> hello (Edit Distance: 1 )

Final Corrected Query:
hello
